In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# AI Transpiler 패스
AI 기반 Transpiler 패스는 일부 트랜스파일링 작업에서 "전통적인" Qiskit 패스를 대체하여 사용할 수 있는 패스입니다. 기존 휴리스틱 알고리즘(낮은 깊이 및 CNOT 수 등)보다 더 나은 결과를 자주 도출하며, 불리언 만족도 해석기(Boolean satisfiability solver)와 같은 최적화 알고리즘보다 훨씬 빠릅니다. AI Transpiler 패스는 로컬 환경에서 실행하거나, IBM Quantum&reg; Premium Plan, Flex Plan, 또는 On-Prem (IBM Quantum Platform API 경유) Plan에 속해 있는 경우 Qiskit Transpiler Service를 통해 클라우드에서 실행할 수 있습니다.

> **Note:** AI 기반 Transpiler 패스는 베타 릴리스 상태로, 변경될 수 있습니다.
>     피드백이 있거나 개발팀에 연락하고 싶은 경우 이 [Qiskit Slack Workspace 채널](https://qiskit.slack.com/archives/C06KF8YHUAU)을 이용해 주세요.

현재 이용 가능한 패스는 다음과 같습니다:

**라우팅 패스**
 - `AIRouting`: 레이아웃 선택 및 Circuit 라우팅

**Circuit 합성 패스**
 - `AICliffordSynthesis`: Clifford Circuit 합성
 - `AILinearFunctionSynthesis`: 선형 함수 Circuit 합성
 - `AIPermutationSynthesis`: 순열 Circuit 합성
 - `AIPauliNetworkSynthesis`: Pauli Network Circuit 합성

AI Transpiler 패스를 사용하려면 먼저 [`qiskit-ibm-transpiler` 패키지를 설치](/guides/qiskit-transpiler-service#install-transpiler-service)하세요. 사용 가능한 다양한 옵션에 대한 자세한 내용은 [qiskit-ibm-transpiler API 문서](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler)를 참고하세요.

## 로컬 또는 클라우드에서 AI Transpiler 패스 실행
로컬 환경에서 무료로 AI 기반 Transpiler 패스를 사용하고 싶다면, 아래와 같이 추가 의존성을 포함하여 `qiskit-ibm-transpiler`를 설치하세요:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

이러한 추가 의존성 없이는 AI 기반 Transpiler 패스가 Qiskit Transpiler Service를 통해 클라우드에서 실행됩니다(IBM Quantum Premium Plan, Flex Plan, 또는 On-Prem (IBM Quantum Platform API 경유) Plan 사용자만 이용 가능). 추가 의존성을 설치하면 AI 기반 Transpiler 패스의 기본 실행 모드는 로컬 머신을 사용하는 방식으로 전환됩니다.

## AI 라우팅 패스
`AIRouting` 패스는 레이아웃 단계와 라우팅 단계 모두로 동작합니다. 다음과 같이 `PassManager` 내에서 사용할 수 있습니다:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

여기서 `backend`는 라우팅할 커플링 맵을 결정하고, `optimization_level`(1, 2, 또는 3)은 처리 과정에서 투입할 계산 노력을 결정합니다(값이 높을수록 일반적으로 더 나은 결과를 제공하지만 더 오래 걸립니다). `layout_mode`는 레이아웃 선택 방식을 지정합니다.
`layout_mode`에는 다음과 같은 옵션이 있습니다:

- `keep`: 이전 Transpiler 패스에서 설정한 레이아웃을 유지합니다(설정되지 않은 경우 trivial 레이아웃을 사용합니다). 일반적으로 Circuit을 장치의 특정 Qubit에서 실행해야 하는 경우에만 사용됩니다. 최적화 여지가 줄어들어 결과가 좋지 않은 경우가 많습니다.
- `improve`: 이전 Transpiler 패스에서 설정한 레이아웃을 시작점으로 사용합니다. 레이아웃에 대한 좋은 초기 추정값이 있는 경우, 예를 들어 장치의 커플링 맵을 대략적으로 따르는 방식으로 구성된 Circuit에 유용합니다. 또한 특정 레이아웃 패스와 `AIRouting` 패스를 조합하여 시도하고 싶을 때도 유용합니다.
- `optimize`: 기본 모드입니다. 레이아웃 추정이 어려운 일반적인 Circuit에 가장 잘 작동합니다. 이 모드는 이전 레이아웃 선택을 무시합니다.
- `local_mode`: 이 플래그는 `AIRouting` 패스가 실행되는 위치를 결정합니다. `False`이면 `AIRouting`이 Qiskit Transpiler Service를 통해 원격으로 실행됩니다. `True`이면 패키지가 로컬 환경에서 패스를 실행하려고 시도하며, 필요한 의존성이 없을 경우 클라우드 모드로 폴백합니다.

## AI Circuit 합성 패스
AI Circuit 합성 패스를 사용하면 다양한 Circuit 유형([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network)의 일부를 재합성하여 최적화할 수 있습니다. 합성 패스를 사용하는 일반적인 방법은 다음과 같습니다:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

합성은 장치의 커플링 맵을 준수합니다. 다른 라우팅 패스 이후에 Circuit을 방해하지 않고 안전하게 실행할 수 있으므로 전체 Circuit은 여전히 장치 제약 조건을 따릅니다. 기본적으로 합성 패스는 합성된 서브 Circuit이 원래 Circuit보다 개선된 경우(현재는 CNOT 수만 확인)에만 원래 서브 Circuit을 교체하지만, `replace_only_if_better=False`로 설정하여 항상 교체하도록 강제할 수 있습니다.

`qiskit_ibm_transpiler.ai.synthesis`에서 가져올 수 있는 합성 패스는 다음과 같습니다:

- *AICliffordSynthesis*: [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) Circuit(`H`, `S`, `CX` Gate 블록)의 합성. 현재 최대 9개의 Qubit 블록까지 지원합니다.
- *AILinearFunctionSynthesis*: [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) Circuit(`CX` 및 `SWAP` Gate 블록)의 합성. 현재 최대 9개의 Qubit 블록까지 지원합니다.
- *AIPermutationSynthesis*: [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) Circuit(`SWAP` Gate 블록)의 합성. 현재 65, 33, 27개의 Qubit 블록에서 사용 가능합니다.
- *AIPauliNetworkSynthesis*: Pauli Network Circuit(`H`, `S`, `SX`, `CX`, `RX`, `RY`, `RZ` Gate 블록)의 합성. 현재 최대 6개의 Qubit 블록까지 지원합니다.

지원하는 블록의 크기는 점차적으로 늘려 나갈 예정입니다.

모든 패스는 스레드 풀을 사용하여 여러 요청을 병렬로 전송합니다. 기본적으로 최대 스레드 수는 코어 수에 4를 더한 값입니다(`ThreadPoolExecutor` Python 객체의 기본값). 그러나 패스 인스턴스화 시 `max_threads` 인수를 사용하여 직접 값을 설정할 수 있습니다. 예를 들어, 다음 코드는 최대 20개의 스레드를 사용하도록 `AILinearFunctionSynthesis` 패스를 인스턴스화합니다.